# Book Recommendation Engine using K-Nearest Neighbors (KNN)
Run the original freeCodeCamp notebook's setup/data-loading cells first, then run this notebook. It expects `books` and `ratings` DataFrames to exist. If you are using this as a standalone notebook, open the freeCodeCamp notebook and copy its first three setup/data-loading cells above the implementation.

In [37]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors


In [38]:
# Download the Book-Crossing dataset zip file and unzip it
!wget -q https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
!unzip -q -o book-crossings.zip

# Load the CSV datasets into 'books' and 'ratings' variables with correct configurations
books = pd.read_csv(
    'BX-Books.csv',
    sep=';',
    header=0,
    names=['ISBN', 'title', 'author'],
    usecols=['ISBN', 'title', 'author'],
    dtype={'ISBN': 'str', 'title': 'str', 'author': 'str'},
    encoding='ISO-8859-1'
)

ratings = pd.read_csv(
    'BX-Book-Ratings.csv',
    sep=';',
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'int32'},
    encoding='ISO-8859-1'
)

print(f"Data successfully loaded!")
print(f"Books DataFrame shape: {books.shape}")
print(f"Ratings DataFrame shape: {ratings.shape}")

Data successfully loaded!
Books DataFrame shape: (271379, 3)
Ratings DataFrame shape: (1149780, 3)


In [39]:
# Preview the first 5 rows of the books DataFrame
print("Books DataFrame:")
display(books.head())

# Preview the first 5 rows of the ratings DataFrame
print("\nRatings DataFrame:")
display(ratings.head())

Books DataFrame:


,ISBN,title,author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber



Ratings DataFrame:


,user,isbn,rating
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0
3,276729,052165615X,3
4,276729,0521795028,6


In [40]:
# Check for missing values in the books DataFrame
print("Missing values in books DataFrame:")
display(books.isnull().sum())

# Check for missing values in the ratings DataFrame
print("\nMissing values in ratings DataFrame:")
display(ratings.isnull().sum())

Missing values in books DataFrame:


,0
ISBN,0
title,0
author,2



Missing values in ratings DataFrame:


,0
user,0
isbn,0
rating,0


In [41]:
# Calculate and display the filtering process explicitly
print(f"Original ratings count: {len(ratings)}")

# Filter users with at least 200 ratings
user_counts = ratings['user'].value_counts()
ratings_filtered = ratings[ratings['user'].isin(user_counts[user_counts >= 200].index)]
print(f"Ratings count after filtering users (< 200 ratings removed): {len(ratings_filtered)}")

# Filter books with at least 100 ratings
book_counts = ratings_filtered['isbn'].value_counts()
ratings_filtered = ratings_filtered[ratings_filtered['isbn'].isin(book_counts[book_counts >= 100].index)]
print(f"Ratings count after filtering books (< 100 ratings removed): {len(ratings_filtered)}")

Original ratings count: 1149780
Ratings count after filtering users (< 200 ratings removed): 527556
Ratings count after filtering books (< 100 ratings removed): 13793


In [42]:
# Download the Book-Crossing dataset zip file
!wget -q https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
!unzip -q -o book-crossings.zip
print("Dataset downloaded and extracted successfully.")

Dataset downloaded and extracted successfully.


In [43]:
# Load the CSV datasets into pandas DataFrames with appropriate encoding and delimiters
books = pd.read_csv(
    'BX-Books.csv',
    sep=';',
    header=0,
    names=['ISBN', 'title', 'author'],
    usecols=['ISBN', 'title', 'author'],
    dtype={'ISBN': 'str', 'title': 'str', 'author': 'str'},
    encoding="ISO-8859-1"
)

ratings = pd.read_csv(
    'BX-Book-Ratings.csv',
    sep=';',
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'int32'},
    encoding="ISO-8859-1"
)

print("Books shape:", books.shape)
print("Ratings shape:", ratings.shape)

Books shape: (271379, 3)
Ratings shape: (1149780, 3)


In [44]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors

def _find_column(frame, candidates):
    """Return the first matching column name, ignoring capitalization."""
    lookup = {str(column).strip().lower(): column for column in frame.columns}
    for candidate in candidates:
        if candidate.lower() in lookup:
            return lookup[candidate.lower()]
    raise KeyError(
        f"Could not find any of these columns: {candidates}. "
        f"Available columns: {list(frame.columns)}"
    )

def build_recommendation_model(books, ratings):
    """Filter sparse users/books, build the ratings matrix, and fit KNN."""
    user_col = _find_column(ratings, ["User-ID", "user_id", "user"])
    rating_isbn_col = _find_column(ratings, ["ISBN", "isbn"])
    rating_value_col = _find_column(ratings, ["Book-Rating", "rating", "book_rating"])
    books_isbn_col = _find_column(books, ["ISBN", "isbn"])
    title_col = _find_column(books, ["Book-Title", "title", "book_title"])

    ratings_work = ratings[[user_col, rating_isbn_col, rating_value_col]].copy()
    books_work = books[[books_isbn_col, title_col]].copy()
    ratings_work.columns = ["user_id", "isbn", "rating"]
    books_work.columns = ["isbn", "title"]

    # Clean ISBNs and titles
    ratings_work["isbn"] = ratings_work["isbn"].str.strip().str.upper()
    books_work["isbn"] = books_work["isbn"].str.strip().str.upper()
    books_work["title"] = books_work["title"].str.strip()

    # 1. Filter users with at least 200 ratings
    user_counts = ratings_work["user_id"].value_counts()
    ratings_work = ratings_work[
        ratings_work["user_id"].isin(user_counts[user_counts >= 200].index)
    ]

    # 2. Map ratings to titles to consolidate editions
    title_lookup = books_work.drop_duplicates("isbn").set_index("isbn")["title"]
    ratings_work = ratings_work.join(title_lookup, on="isbn", how="inner")

    # 3. Filter books (by title) with at least 100 ratings
    book_counts = ratings_work["title"].value_counts()
    ratings_work = ratings_work[
        ratings_work["title"].isin(book_counts[book_counts >= 100].index)
    ]

    # Pivot table with 'title' as index to consolidate multiple ISBNs
    ratings_matrix = ratings_work.pivot_table(
        index="title", columns="user_id", values="rating", fill_value=0
    )

    # Fit KNN model directly on the consolidated title-based pivot table
    model = NearestNeighbors(metric="cosine", algorithm="brute")
    model.fit(ratings_matrix)
    return ratings_matrix, model

def get_recommends(book):
    """Return [input_title, [[similar_title, cosine_distance], ...]]."""
    if "books" not in globals() or "ratings" not in globals():
        raise RuntimeError(
            "Run the notebook's data-loading cells first so `books` and `ratings` exist."
        )

    global _recommendation_matrix, _knn_model
    if (
        "_recommendation_matrix" not in globals()
        or "_knn_model" not in globals()
    ):
        _recommendation_matrix, _knn_model = build_recommendation_model(
            globals()["books"], globals()["ratings"]
        )

    # Find matches by case-insensitive title lookup
    matches = _recommendation_matrix.index[
        _recommendation_matrix.index.astype(str).str.lower() == str(book).lower().strip()
    ]
    if len(matches) == 0:
        raise ValueError(f"Book title not found after filtering: {book!r}")

    target_title = matches[0]
    query_vector = _recommendation_matrix.loc[[target_title]]
    distances, indices = _knn_model.kneighbors(
        query_vector, n_neighbors=min(6, len(_recommendation_matrix))
    )

    recommendations = []
    for i in range(1, len(indices.flatten())):
        recommended_title = _recommendation_matrix.index[indices.flatten()[i]]
        recommendations.append([recommended_title, float(distances.flatten()[i])])

    # Reverse so the nearest recommendations are listed in the required test assertions order
    recommendations.reverse()
    return [book, recommendations]

## Test
After running the setup/data-loading cells from the original challenge, run:

In [45]:
import os
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# Ensure dataset is present
if not os.path.exists('BX-Books.csv') or not os.path.exists('BX-Book-Ratings.csv'):
    print("[Debug] Downloading and extracting the Book-Crossing dataset...")
    os.system("wget -q https://cdn.freecodecamp.org/project-data/books/book-crossings.zip")
    os.system("unzip -q -o book-crossings.zip")

print("[Debug] Loading CSV datasets...")
books = pd.read_csv(
    'BX-Books.csv',
    sep=';',
    header=0,
    names=['ISBN', 'title', 'author'],
    usecols=['ISBN', 'title', 'author'],
    dtype={'ISBN': 'str', 'title': 'str', 'author': 'str'},
    encoding='ISO-8859-1'
)
ratings = pd.read_csv(
    'BX-Book-Ratings.csv',
    sep=';',
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'int32'},
    encoding='ISO-8859-1'
)

# Clear cached variables
globals().pop('_recommendation_matrix', None)
globals().pop('_knn_model', None)

def clean_title(title_series):
    # Strip quotes, multiple spaces, and outer whitespaces
    return title_series.astype(str).str.replace(r'["\']', '', regex=True).str.replace(r'\s+', ' ', regex=True).str.strip()

def build_recommendation_model(books, ratings):
    df_ratings = ratings.copy()
    df_books = books.copy()

    # Normalize columns
    df_ratings.columns = [str(c).lower().strip() for c in df_ratings.columns]
    df_books.columns = [str(c).lower().strip() for c in df_books.columns]

    # Normalize standard key variables
    df_ratings['isbn'] = df_ratings['isbn'].str.strip().str.upper()
    df_books['isbn'] = df_books['isbn'].str.strip().str.upper()
    df_books['title'] = clean_title(df_books['title'])

    print(f"[Debug] Initial ratings count: {len(df_ratings)}")

    # 1. Filter active users (>= 200 ratings)
    user_counts = df_ratings['user'].value_counts()
    active_users = user_counts[user_counts >= 200].index
    df_ratings = df_ratings[df_ratings['user'].isin(active_users)]
    print(f"[Debug] Ratings after active user filter (>= 200): {len(df_ratings)}")


    # 2. Filter active books (>= 100 ratings from all users to keep popular books)
    # The standard freeCodeCamp requirement is that the final dataset contains books with >= 100 ratings.
    raw_book_counts = ratings['isbn'].str.strip().str.upper().value_counts()
    popular_isbns = raw_book_counts[raw_book_counts >= 100].index
    df_ratings = df_ratings[df_ratings['isbn'].isin(popular_isbns)]
    print(f"[Debug] Ratings after active book filter (>= 100): {len(df_ratings)}")

    # 3. Map ratings to book titles
    books_clean = df_books.drop_duplicates(subset=['isbn'])
    df_ratings = df_ratings.join(books_clean.set_index('isbn')['title'], on='isbn', how='inner')
    print(f"[Debug] Ratings after joining with books catalog: {len(df_ratings)}")

    # Verify target book presence
    target_query = "The Queen of the Damned (Vampire Chronicles (Paperback))"
    has_target = df_ratings['title'].str.lower().eq(target_query.lower()).any()
    print(f"[Debug] Target title present in filtered ratings: {has_target}")

    # 4. Create final pivot matrix
    pivot_df = df_ratings.pivot_table(index='title', columns='user', values='rating', fill_value=0)
    print(f"[Debug] Final pivot matrix shape: {pivot_df.shape}")

    # Fit KNN model
    model = NearestNeighbors(metric='cosine', algorithm='brute')
    model.fit(pivot_df)
    return pivot_df, model

def get_recommends(book):
    global _recommendation_matrix, _knn_model
    if '_recommendation_matrix' not in globals() or '_knn_model' not in globals():
        _recommendation_matrix, _knn_model = build_recommendation_model(books, ratings)

    # Case-insensitive title lookup
    target_clean = str(book).lower().strip()
    matches = _recommendation_matrix.index[
        _recommendation_matrix.index.astype(str).str.lower() == target_clean
    ]
    if len(matches) == 0:
        raise ValueError(f"Book title not found: {book!r}")

    target_title = matches[0]
    query_vector = _recommendation_matrix.loc[[target_title]]
    distances, indices = _knn_model.kneighbors(query_vector, n_neighbors=6)

    recommendations = []
    for i in range(1, len(indices.flatten())):
        rec_title = _recommendation_matrix.index[indices.flatten()[i]]
        rec_dist = distances.flatten()[i]
        recommendations.append([rec_title, float(rec_dist)])

    recommendations.reverse()
    return [book, recommendations]

_recommendation_matrix, _knn_model = build_recommendation_model(books, ratings)
recommendations = get_recommends("The Queen of the Damned (Vampire Chronicles (Paperback))")
display(recommendations)

[Debug] Loading CSV datasets...
[Debug] Initial ratings count: 1149780
[Debug] Ratings after active user filter (>= 200): 527556
[Debug] Ratings after active book filter (>= 100): 49810
[Debug] Ratings after joining with books catalog: 49546
[Debug] Target title present in filtered ratings: True
[Debug] Final pivot matrix shape: (673, 888)


['The Queen of the Damned (Vampire Chronicles (Paperback))',
 [['Catch 22', 0.7939835419270879],
  ['The Witching Hour (Lives of the Mayfair Witches)', 0.7448657003312193],
  ['Interview with the Vampire', 0.7345068863988313],
  ['The Tale of the Body Thief (Vampire Chronicles (Paperback))',
   0.5376338446489461],
  ['The Vampire Lestat (Vampire Chronicles, Book II)', 0.5178411864186412]]]

In [46]:
import pandas as pd

# Prepare the data from our recommendations list
rec_book, rec_list = recommendations

df_export = pd.DataFrame(rec_list, columns=['Recommended Book Title', 'Cosine Distance'])
df_export.insert(0, 'Query Book', rec_book)

# Save to CSV
csv_filename = 'recommendations.csv'
df_export.to_csv(csv_filename, index=False)
print(f"Recommendations successfully saved to {csv_filename}!")
display(df_export)

Recommendations successfully saved to recommendations.csv!


,Query Book,Recommended Book Title,Cosine Distance
0,The Queen of the Damned (Vampire Chronicles (P...,Catch 22,0.793984
1,The Queen of the Damned (Vampire Chronicles (P...,The Witching Hour (Lives of the Mayfair Witches),0.744866
2,The Queen of the Damned (Vampire Chronicles (P...,Interview with the Vampire,0.734507
3,The Queen of the Damned (Vampire Chronicles (P...,The Tale of the Body Thief (Vampire Chronicles...,0.537634
4,The Queen of the Damned (Vampire Chronicles (P...,"The Vampire Lestat (Vampire Chronicles, Book II)",0.517841


In [47]:
# Check if the title exists in the books dataset and find its ISBNs
target_title = "The Queen of the Damned (Vampire Chronicles (Paperback))"
matching_books = books[books['title'].str.contains("Queen of the Damned", case=False, na=False)]
print("Matching Books in BX-Books.csv:")
display(matching_books)

# Check how many ratings exist for these ISBNs
if not matching_books.empty:
    isbns = matching_books['ISBN'].str.strip().str.upper().tolist()
    matching_ratings = ratings[ratings['isbn'].str.strip().str.upper().isin(isbns)]
    print(f"\nTotal ratings for these ISBNs: {len(matching_ratings)}")

    # Check how many ratings come from users with at least 200 reviews
    user_counts = ratings['user'].value_counts()
    active_users = user_counts[user_counts >= 200].index
    active_ratings = matching_ratings[matching_ratings['user'].isin(active_users)]
    print(f"Ratings from users with >= 200 reviews: {len(active_ratings)}")

Matching Books in BX-Books.csv:


,ISBN,title,author
2527,0345351525,The Queen of the Damned (Vampire Chronicles (P...,Anne Rice
15190,0394558235,"Queen of the Damned (Vampire Chronicles, Book ...",Anne Rice
49078,0345362608,Queen of the Damned,Anne Rice
72840,0833563505,The Queen of the Damned (Vampire Chronicles (P...,Anne Rice
115611,0394573188,The Queen of the Damned/Audio Cassette (Norton...,Anne Rice
198823,0345419626,Queen of the Damned (Vampire Chronicles (Paper...,Anne Rice
205885,3442098432,Der Konigin der Verdmmen / Queen of the Damned,Anne Rice



Total ratings for these ISBNs: 323
Ratings from users with >= 200 reviews: 115


In [48]:
# Validate recommendations and display the results
try:
    recommendations = get_recommends("The Queen of the Damned (Vampire Chronicles (Paperback))")
    print("Recommendations successful:\n")
    import pprint
    pprint.pprint(recommendations)
except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure you run the cell that downloads the files and the cell that loads them into 'books' and 'ratings' first.")

Recommendations successful:

['The Queen of the Damned (Vampire Chronicles (Paperback))',
 [['Catch 22', 0.7939835419270879],
  ['The Witching Hour (Lives of the Mayfair Witches)', 0.7448657003312193],
  ['Interview with the Vampire', 0.7345068863988313],
  ['The Tale of the Body Thief (Vampire Chronicles (Paperback))',
   0.5376338446489461],
  ['The Vampire Lestat (Vampire Chronicles, Book II)', 0.5178411864186412]]]
